<a href="https://colab.research.google.com/github/crjacinto/ML-IQ/blob/main/ML_IQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introducción al Machine Learning en Ingeniería Química
## Taller práctico · Termodinámica de Procesos

**Objetivo:** construir modelos de regresión para predecir la potencia generada en turbinas a partir de datos de operación, y entender por qué la calidad de los datos y el criterio físico pesan más que el algoritmo.

**Datos utilizados**

1. `Datos_CH.csv`: turbina de vapor de extracción-condensación de un ingenio azucarero. Un registro por día (probablemente promedios diarios) durante la zafra 2025–2026.
2. `gt_2011.csv`: turbina de gas de una central de ciclo combinado en Turquía. Promedios horarios del año 2011, ordenados cronológicamente (UCI Machine Learning Repository).

**Estructura**

1. Colab y Python básico
2. Set 1: datos del ingenio
3. Set 2: turbina de gas (UCI)
4. Comparación con modelos termodinámicos
5. Conclusiones y tarea

## 1. Colab y Python básico

**Cómo usar este notebook**

- Una celda se ejecuta con `Shift + Enter` o con el botón ▶ a su izquierda.
- Las celdas deben ejecutarse **en orden**, de arriba hacia abajo: muchas usan variables definidas en celdas anteriores.
- Si algo falla de forma extraña, usa *Entorno de ejecución → Reiniciar sesión* y vuelve a ejecutar desde el inicio.
- Para guardar tus cambios usa *Archivo → Guardar una copia en Drive*.

In [ ]:
# Librerías principales (todas vienen preinstaladas en Colab)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Modelos y herramientas de scikit-learn
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Semilla fija: así todos obtenemos los mismos resultados
SEMILLA = 42

# Tamaño por defecto de las figuras
plt.rcParams["figure.figsize"] = (10, 4)

### Ubicación de los datos

Los archivos están en un repositorio de GitHub. Si la descarga falla, puedes subirlos manualmente: en el panel izquierdo de Colab, haz clic en el ícono de carpeta → *Subir*, y reemplaza las URLs de la siguiente celda por los nombres de archivo (`"Datos_CH.csv"` y `"gt_2011.csv"`).

In [ ]:
# TODO: reemplazar USUARIO por el nombre de la cuenta de GitHub
URL_BASE = "https://github.com/crjacinto/ML-IQ.git"

URL_INGENIO = URL_BASE + "Datos_CH.csv"
URL_TURBINA = URL_BASE + "gt_2011.csv"

### Funciones auxiliares

Definimos dos funciones que usaremos varias veces durante el taller: una para calcular las métricas de error y otra para comparar gráficamente valores reales y predichos.

- **MAE** (error absoluto medio): error promedio, en las mismas unidades que la variable.
- **RMSE** (raíz del error cuadrático medio): similar al MAE, pero penaliza más los errores grandes.
- **R²**: fracción de la variabilidad de los datos que explica el modelo (1 = perfecto).

In [ ]:
def evaluar_modelo(y_real, y_pred, nombre="modelo"):
    """Calcula y muestra MAE, RMSE y R² de un modelo."""
    mae = mean_absolute_error(y_real, y_pred)
    rmse = np.sqrt(mean_squared_error(y_real, y_pred))
    r2 = r2_score(y_real, y_pred)
    print(f"{nombre}: MAE = {mae:.2f} | RMSE = {rmse:.2f} | R² = {r2:.3f}")
    return {"modelo": nombre, "MAE": mae, "RMSE": rmse, "R2": r2}


def graficar_real_vs_pred(y_real, y_pred, titulo="", unidad=""):
    """Grafica predicción vs. valor real; la diagonal es la predicción perfecta."""
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(y_real, y_pred, alpha=0.6)
    limites = [min(y_real.min(), y_pred.min()), max(y_real.max(), y_pred.max())]
    ax.plot(limites, limites, "k--", label="Predicción perfecta")
    ax.set_xlabel(f"Valor real {unidad}")
    ax.set_ylabel(f"Valor predicho {unidad}")
    ax.set_title(titulo)
    ax.legend()
    plt.show()

### Mini repaso de pandas

`pandas` organiza los datos en tablas llamadas **DataFrames**, similares a una hoja de Excel: cada columna es una variable y cada fila un registro. Hagamos un primer contacto con los datos del ingenio.

In [ ]:
# Lectura directa, sin ninguna opción adicional
df_crudo = pd.read_csv(URL_INGENIO)

# Primeras 5 filas
df_crudo.head()

In [ ]:
# Tipo de dato y cantidad de valores no nulos por columna
df_crudo.info()

**Observa:** la columna `Fecha` aparece como `object` (texto), no como fecha. Además, todas las variables numéricas tienen menos valores no nulos que el total de filas: hay datos faltantes. Lo corregiremos en la Sección 2.

In [ ]:
# Estadísticas descriptivas (.T transpone la tabla para leerla mejor)
df_crudo.describe().T

In [ ]:
# Seleccionar una columna
potencia = df_crudo["Potencia[Mw]"]

# Filtrar filas que cumplen una condición
dias_alta_potencia = df_crudo[df_crudo["Potencia[Mw]"] > 40]
print(f"Días con potencia mayor a 40 MW: {len(dias_alta_potencia)} de {len(df_crudo)}")

In [ ]:
# Gráfico rápido de una columna
df_crudo["Potencia[Mw]"].plot(title="Potencia generada", ylabel="MW", xlabel="Fila")
plt.show()